# Stale-State J-Lens Probe — Qwen3-8B

**Team tommy-hjra · Algoverse AI Research · Axis 2 (State Tracking & Belief Update)**

Does Qwen3-8B internally encode the **updated** location while emitting the **stale** one?

| | |
|---|---|
| **Input** | `.jsonl` from `stale-state-data-generator` (`prompt`, `correct_state`, `stale_state`, `compression`, `entity_type`, …) |
| **Instruments** | logit lens + **J-Lens** (Jacobian lens) at every residual depth, plus per-head attention at the readout position |
| **Output** | `layers.csv`, `examples.csv`, `heads.csv`, `records.jsonl` for offline analysis |

**Runtime:** `Runtime ▸ Change runtime type ▸ T4 GPU` (works in 4-bit) or A100 (bf16, exact J-Lens).

**Upload first:** `jlens_probe.py`, `analyze_results.py`, and your `.jsonl` (drag into the file browser on the left, or use the upload cell below).

In [ ]:
#@title 1 · GPU check
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n{torch.cuda.get_device_name(0)} | {gb:.1f} GB | sm_{cap[0]}{cap[1]} | bf16={cap[0] >= 8}")
    print("suggested --load-in:", "bf16" if (gb >= 40 and cap[0] >= 8) else ("8bit" if gb >= 24 and cap[0] >= 8 else "4bit"))
else:
    print("NO GPU — set Runtime ▸ Change runtime type ▸ T4 GPU")

In [ ]:
#@title 2 · Install
# transformers >= 4.51 is REQUIRED: that is the release that added the Qwen3 architecture.
!pip -q install -U "transformers>=4.51.0" "accelerate>=0.34" bitsandbytes pandas matplotlib

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
#@title 3 · Get the code + data
#@markdown Either drag the files into Colab's file browser, or run this to upload them.
import os

NEEDED = ["jlens_probe.py"]
if not all(os.path.exists(f) for f in NEEDED):
    from google.colab import files
    print("Upload jlens_probe.py, analyze_results.py and your .jsonl:")
    files.upload()

# Point this at your dataset. The generator writes to data/staleness_5k.jsonl.
JSONL = "data/sample_probe_input.jsonl"  #@param {type:"string"}
if not os.path.exists(JSONL):
    cand = [f for f in os.listdir(".") if f.endswith(".jsonl")]
    if cand:
        JSONL = cand[0]
        print(f"using {JSONL}")
    else:
        raise FileNotFoundError("no .jsonl found — upload one")

!head -c 400 "$JSONL"; echo; echo "---"; wc -l "$JSONL"

## 4 · Dry run — check the stimuli before spending GPU time

Loads **only the tokenizer**. Verifies that

* every room name maps to a *distinct* first token (otherwise rank metrics are meaningless),
* the constructed prompt really ends at the position where the room must be predicted,
* both room names are locatable in the text for the attention analysis.

In [ ]:
!python jlens_probe.py --jsonl "$JSONL" --dry-run --limit 8

## 5 · Run the probe

`--self-check` replays the truncated stack from layer 0 and asserts it reproduces the real
logits — do this once per session; it validates the cache slicing, RoPE positions and
layer plumbing that J-Lens rides on.

Start with `--limit 20` to time it, then scale up. Useful knobs:

| flag | why |
|---|---|
| `--layers 0-36:2` | probe every other depth — roughly halves runtime |
| `--filter-compression aggressive` | the condition most likely to induce stale fallback |
| `--target-mode generate` | let the model answer freely and read out where it names a room |
| `--jlens-mode fd` | force finite differences (always safe under quantisation) |
| `--dump-heads` | also write the per-head attention table |

In [ ]:
OUT  = "outputs/run1"
ARGS = ("--limit 20 "               # raise once you have timed one example
        "--load-in auto "           # auto -> bf16 on A100, 4bit on T4
        "--prompt-format chat "     # Qwen3 chat template, <think> suppressed
        "--jlens-mode auto "        # autograd if unquantised, else finite differences
        "--jlens-baseline zero "
        "--dump-heads "             # per-head attention table
        "--self-check")             # verify the truncated-stack replay

!python jlens_probe.py --jsonl "$JSONL" --out-dir "$OUT" $ARGS

## 6 · Analysis

`T2` is the headline: of the trials where the model **emitted the stale room**, in how many
was the **correct** room already outranking it somewhere inside?

* high  → **retrieval failure** (state encoded, not used) — the Axis 2 hypothesis
* low   → **representation failure** (state never encoded)

In [ ]:
!python analyze_results.py "$OUT" --plot

In [ ]:
#@title 7 · Depth profile figure
from IPython.display import Image, display
import os
p = os.path.join(OUT, "depth_profile.png")
display(Image(p)) if os.path.exists(p) else print("run cell 6 with --plot first")

In [ ]:
#@title 8 · Inspect one example layer by layer
import json, pandas as pd

recs = [json.loads(l) for l in open(f"{OUT}/records.jsonl")]
stale = [r for r in recs if r["output_is_stale"]]
r = (stale or recs)[0]

print(f"{r['example_id']} | level {r['complexity_level']} | compression={r['compression']} "
      f"| entity_type={r['entity_type']}")
print(f"correct = {r['correct_state']!r} ({r['correct_token']})   "
      f"stale = {r['stale_state']!r} ({r['stale_token']})")
print(f"model emitted: {r['final_top1_token']!r}  ->  {r['outcome'].upper()}\n")
print("..." + r["prompt_text"][-260:])

df = pd.DataFrame(r["layers"])[[
    "layer", "logitlens_rank_correct", "logitlens_rank_stale",
    "jlens_rank_correct", "jlens_rank_stale", "jlens_top1_token",
    "attn_correct_update_mass", "attn_stale_anchor_mass"]]
display(df.style.background_gradient(subset=["jlens_rank_correct"], cmap="RdYlGn_r"))

In [ ]:
#@title 9 · Download results
import shutil
from google.colab import files
shutil.make_archive("jlens_results", "zip", OUT)
files.download("jlens_results.zip")

---
### Troubleshooting

| symptom | fix |
|---|---|
| `KeyError: 'qwen3'` | `transformers` too old — needs ≥ 4.51.0, then restart the runtime |
| OOM on T4 | `--load-in 4bit`, and `--layers 0-36:4` to probe fewer depths |
| self-check `max\|Δlogit\|` large | transformers changed the decoder-layer signature; pin `transformers==4.51.3` |
| all `attn_*` columns NaN | attention weights unavailable — harmless; add `--no-attention` to skip the pass |
| autograd JVP warning | expected under 4-bit/8-bit; it falls back to finite differences automatically |
| `first_token_collision=True` rows | two rooms share a first token — drop those rows before analysis |